# 01. Introducción a LLM Agents

**Nivel:** 🟢 Principiante  
**Tiempo estimado:** 90-120 minutos  
**Prerequisitos:** Familiaridad básica con LLMs y APIs

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Explicar la diferencia fundamental entre un LLM y un Agente basado en LLM
- Identificar los componentes clave de un agente (percepción, razonamiento, acción)
- Implementar un loop de agente simple desde cero
- Comprender el ciclo observación → pensamiento → acción → observación
- Configurar y usar APIs de LLMs (OpenAI, Anthropic, o modelos locales)
- **Implementar 5 funciones core de sistemas agénticos (100 puntos)**

---

## 📋 Tabla de Contenidos

1. [Motivación: ¿Por qué Agentes?](#1-motivacion)
2. [Intuición Visual: Anatomía de un Agente](#2-intuicion-visual)
3. [Fundamentos Matemáticos](#3-fundamentos-matematicos)
4. [Implementación Desde Cero](#4-implementacion-desde-cero)
5. [🎓 Ejercicios Prácticos Guiados (100 pts)](#5-ejercicios-graded)
6. [Comparación de Frameworks](#6-frameworks)
7. [Ejercicios Avanzados (Opcionales)](#7-ejercicios-avanzados)
8. [📄 Papers y Referencias](#8-papers)
9. [💡 Best Practices y Producción](#9-best-practices)
10. [📍 Navegación y Próximos Pasos](#10-navegacion)

---


<a id="1-motivacion"></a>
## 1. Motivación: ¿Por qué Agentes?

### El Problema con LLMs Puros

Imagina que le preguntas a ChatGPT: *"¿Cuál es el clima en Madrid ahora mismo?"*

El LLM responderá algo como: *"Lo siento, no tengo acceso a información en tiempo real..."* 

**¿Por qué?** Porque un LLM puro:
- Solo tiene conocimiento hasta su fecha de entrenamiento
- No puede acceder a APIs o herramientas externas
- No puede ejecutar acciones en el mundo real
- Solo puede generar texto basado en su contexto

### La Solución: Agentes

Un **Agente basado en LLM** puede:
1. **Razonar** sobre qué información necesita
2. **Decidir** usar una herramienta (ej: API de clima)
3. **Ejecutar** la acción (llamar a la API)
4. **Observar** el resultado
5. **Razonar nuevamente** con la nueva información
6. **Responder** al usuario con datos actualizados

### Pregunta Guía

**Al final de este notebook responderemos:**
*¿Cómo podemos transformar un LLM pasivo (solo genera texto) en un agente activo (razona y ejecuta acciones)?*


<a id="2-intuicion-visual"></a>
## 2. Intuición Visual: Anatomía de un Agente

### LLM vs Agente: Comparación

```
┌─────────────────────────────────────────────────────────────┐
│                      LLM (Pasivo)                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Input (Prompt) ──────► [LLM] ──────► Output (Text)       │
│                                                             │
│  • Solo procesa texto                                      │
│  • Una pasada, sin iteración                               │
│  • No puede usar herramientas                              │
│  • Conocimiento estático                                   │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    Agente (Activo)                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│           ┌──────────────────────────┐                     │
│           │                          │                     │
│           ▼                          │                     │
│  Observación ──► Razonamiento ──► Acción                  │
│      ▲         (LLM Core)        │                        │
│      │                           │                        │
│      │                           ▼                        │
│      │                      [Herramientas]                │
│      │                      • Web Search                  │
│      │                      • Calculator                  │
│      │                      • Database                    │
│      │                      • Code Exec                   │
│      └──────────────────────────┘                         │
│                                                             │
│  • Loop iterativo                                          │
│  • Usa herramientas externas                               │
│  • Actualiza conocimiento dinámicamente                    │
│  • Toma decisiones y ejecuta acciones                      │
└─────────────────────────────────────────────────────────────┘
```

### Componentes de un Agente

1. **Cerebro (LLM)**: Razona sobre qué hacer
2. **Memoria**: Recuerda interacciones pasadas y contexto
3. **Herramientas**: Capacidades extendidas (APIs, calculadora, etc.)
4. **Control Loop**: Orquesta el ciclo observar-pensar-actuar
5. **Planner** (opcional): Descompone tareas complejas en pasos

Visualizaremos esto con código a continuación.


In [ ]:
# Instalación de dependencias necesarias
# Descomenta si no las tienes instaladas

# !pip install openai anthropic python-dotenv plotly pandas
# Para modelos locales (opcional):
# !pip install transformers torch


In [ ]:
import os
import json
from typing import List, Dict, Optional, Callable
from dataclasses import dataclass
from datetime import datetime

# Para visualizaciones
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Para APIs de LLMs
try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False
    print("⚠️  OpenAI no disponible. Instala con: pip install openai")

try:
    from anthropic import Anthropic
    ANTHROPIC_AVAILABLE = True
except ImportError:
    ANTHROPIC_AVAILABLE = False
    print("⚠️  Anthropic no disponible. Instala con: pip install anthropic")

# Configuración
from dotenv import load_dotenv
load_dotenv()

print("✅ Librerías importadas correctamente")


<a id="3-fundamentos-matematicos"></a>
## 3. Fundamentos Matemáticos: Formalización del Agente

### Definición Formal

Un **agente** puede formalizarse como una función que mapea secuencias de percepciones a acciones:

$$
\begin{align}
\pi: \mathcal{O}^* &\to \mathcal{A} \tag{1} \\
\text{donde: } & \\
\mathcal{O} &: \text{espacio de observaciones} \\
\mathcal{A} &: \text{espacio de acciones} \\
\pi &: \text{política del agente (la "estrategia")} \\
\mathcal{O}^* &: \text{historial de observaciones}
\end{align}
$$

### El Loop del Agente

En cada paso temporal $t$:

$$
\begin{align}
o_t &= \text{observe}(\text{environment}) \tag{2} \\
s_t &= \text{update\_state}(s_{t-1}, o_t) \tag{3} \\
a_t &= \pi(s_t) \tag{4} \\
\text{environment} &= \text{execute}(a_t) \tag{5}
\end{align}
$$

Donde:
- $o_t$: Observación en tiempo $t$
- $s_t$: Estado interno del agente (memoria)
- $a_t$: Acción elegida por la política
- $\pi$: Política implementada por el LLM

### Para LLM Agents

La política $\pi$ es implementada por un LLM:

$$
\begin{align}
\pi(s_t) &= \text{LLM}(\text{prompt}(s_t)) \tag{6} \\
\text{prompt}(s_t) &= \text{template}(\text{system\_msg}, \text{history}_t, \text{tools}) \tag{7}
\end{align}
$$

**Key Insight:**
> La "inteligencia" del agente emerge de cómo construimos el prompt que alimenta al LLM. El diseño del prompt determina qué tan bien razona y actúa el agente.

### Ejemplo Numérico

Supongamos:
- Usuario pregunta: "¿Cuánto es 15% de 340?"
- Agente tiene herramienta: `calculator`

**Paso 1:** $o_0$ = "¿Cuánto es 15% de 340?"  
**Paso 2:** LLM razona → decide usar `calculator(0.15 * 340)`  
**Paso 3:** Herramienta ejecuta → $o_1$ = "51.0"  
**Paso 4:** LLM genera respuesta final → "El 15% de 340 es 51"

Este loop puede iterar múltiples veces según la complejidad de la tarea.


<a id="4-implementacion-desde-cero"></a>
## 4. Implementación Desde Cero: Simple Agent Loop

Implementaremos un agente básico con tres componentes:
1. **LLM Backend** (OpenAI, Anthropic, o simulado)
2. **Herramientas simples** (calculadora, hora actual)
3. **Loop de control**


In [ ]:
@dataclass
class AgentAction:
    """Representa una acción del agente"""
    tool: str  # Nombre de la herramienta a usar
    tool_input: str  # Input para la herramienta
    reasoning: str  # Razonamiento del agente

@dataclass
class Observation:
    """Representa una observación del entorno"""
    content: str  # Contenido de la observación
    timestamp: datetime  # Cuándo ocurrió

class SimpleTool:
    """Clase base para herramientas del agente"""
    def __init__(self, name: str, description: str, func: Callable):
        self.name = name
        self.description = description
        self.func = func
    
    def run(self, input_str: str) -> str:
        """Ejecuta la herramienta con el input dado"""
        try:
            result = self.func(input_str)
            return str(result)
        except Exception as e:
            return f"Error: {str(e)}"

# Definir herramientas simples
def calculator(expression: str) -> float:
    """Calcula expresiones matemáticas simples"""
    # ADVERTENCIA: eval es peligroso en producción, usar solo para demos
    # En producción, usa un parser matemático seguro
    return eval(expression)

def get_current_time(timezone: str = "UTC") -> str:
    """Retorna la hora actual"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Crear herramientas
calculator_tool = SimpleTool(
    name="calculator",
    description="Útil para hacer cálculos matemáticos. Input: expresión matemática en Python.",
    func=calculator
)

time_tool = SimpleTool(
    name="get_time",
    description="Retorna la fecha y hora actual.",
    func=get_current_time
)

print("✅ Herramientas creadas:", [calculator_tool.name, time_tool.name])


In [ ]:
class SimpleAgent:
    """
    Implementación básica de un agente LLM.
    
    El agente:
    1. Recibe una query del usuario
    2. Decide si necesita usar una herramienta
    3. Ejecuta la herramienta si es necesario
    4. Genera una respuesta final
    """
    
    def __init__(self, tools: List[SimpleTool], llm_backend: str = "simulated"):
        """
        Args:
            tools: Lista de herramientas disponibles
            llm_backend: "openai", "anthropic", o "simulated"
        """
        self.tools = {tool.name: tool for tool in tools}
        self.llm_backend = llm_backend
        self.history: List[Dict] = []  # Historial de interacciones
        
        # Configurar cliente LLM
        if llm_backend == "openai" and OPENAI_AVAILABLE:
            self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
            self.model = "gpt-4-turbo-preview"
        elif llm_backend == "anthropic" and ANTHROPIC_AVAILABLE:
            self.client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
            self.model = "claude-3-opus-20240229"
        else:
            self.client = None
            print("⚠️  Usando LLM simulado para demostración")
    
    def _build_prompt(self, query: str) -> str:
        """Construye el prompt para el LLM"""
        tools_desc = "\n".join([
            f"- {name}: {tool.description}" 
            for name, tool in self.tools.items()
        ])
        
        prompt = f"""Eres un asistente útil que puede usar herramientas para responder preguntas.

Herramientas disponibles:
{tools_desc}

Para usar una herramienta, responde EXACTAMENTE en este formato:
RAZONAMIENTO: [tu razonamiento]
HERRAMIENTA: [nombre_herramienta]
INPUT: [input para la herramienta]

Si no necesitas una herramienta, responde directamente.

Pregunta del usuario: {query}

Tu respuesta:"""
        return prompt
    
    def _call_llm(self, prompt: str) -> str:
        """Llama al LLM y retorna la respuesta"""
        if self.llm_backend == "openai" and self.client:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            return response.choices[0].message.content
        
        elif self.llm_backend == "anthropic" and self.client:
            response = self.client.messages.create(
                model=self.model,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.content[0].text
        
        else:
            # LLM simulado para demostración
            return self._simulated_llm(prompt)
    
    def _simulated_llm(self, prompt: str) -> str:
        """Simula un LLM con reglas básicas (solo para demo)"""
        query_lower = prompt.lower()
        
        # Detectar si necesita calculadora
        if any(word in query_lower for word in ['calcular', 'cuánto es', '+', '-', '*', '/', '%']):
            # Intentar extraer expresión matemática
            import re
            # Buscar patrones como "15% de 340" o "2 + 2"
            match = re.search(r'(\d+)%\s+de\s+(\d+)', query_lower)
            if match:
                expr = f"{match.group(1)} * {match.group(2)} / 100"
            else:
                # Buscar expresiones matemáticas simples
                expr_match = re.search(r'[\d+\-*/().\s]+', query_lower)
                expr = expr_match.group(0) if expr_match else "1+1"
            
            return f"""RAZONAMIENTO: Necesito calcular una expresión matemática
HERRAMIENTA: calculator
INPUT: {expr}"""
        
        # Detectar si pregunta por hora
        elif any(word in query_lower for word in ['hora', 'fecha', 'tiempo']):
            return """RAZONAMIENTO: Usuario pregunta por la hora actual
HERRAMIENTA: get_time
INPUT: UTC"""
        
        # Respuesta directa
        return "Hola, puedo ayudarte con cálculos y consultar la hora."
    
    def _parse_action(self, llm_response: str) -> Optional[AgentAction]:
        """Parsea la respuesta del LLM para extraer una acción"""
        if "HERRAMIENTA:" not in llm_response:
            return None
        
        lines = llm_response.strip().split('\n')
        action_dict = {}
        
        for line in lines:
            if line.startswith("RAZONAMIENTO:"):
                action_dict['reasoning'] = line.replace("RAZONAMIENTO:", "").strip()
            elif line.startswith("HERRAMIENTA:"):
                action_dict['tool'] = line.replace("HERRAMIENTA:", "").strip()
            elif line.startswith("INPUT:"):
                action_dict['tool_input'] = line.replace("INPUT:", "").strip()
        
        if 'tool' in action_dict:
            return AgentAction(
                tool=action_dict.get('tool', ''),
                tool_input=action_dict.get('tool_input', ''),
                reasoning=action_dict.get('reasoning', '')
            )
        return None
    
    def run(self, query: str, max_iterations: int = 3, verbose: bool = True) -> str:
        """
        Ejecuta el loop del agente.
        
        Args:
            query: Pregunta del usuario
            max_iterations: Máximo número de iteraciones del loop
            verbose: Si True, imprime pasos intermedios
        
        Returns:
            Respuesta final del agente
        """
        if verbose:
            print(f"\n{'='*60}")
            print(f"🤖 AGENTE EJECUTÁNDOSE")
            print(f"{'='*60}")
            print(f"\n📝 Query: {query}\n")
        
        for iteration in range(max_iterations):
            if verbose:
                print(f"\n--- Iteración {iteration + 1} ---")
            
            # 1. Construir prompt y llamar LLM
            prompt = self._build_prompt(query)
            llm_response = self._call_llm(prompt)
            
            if verbose:
                print(f"\n💭 LLM Response:\n{llm_response}")
            
            # 2. Parsear acción
            action = self._parse_action(llm_response)
            
            # 3. Si no hay acción, retornar respuesta
            if action is None:
                if verbose:
                    print(f"\n✅ Respuesta final (sin herramientas)")
                return llm_response
            
            # 4. Ejecutar herramienta
            if action.tool not in self.tools:
                error_msg = f"Error: Herramienta '{action.tool}' no disponible"
                if verbose:
                    print(f"\n❌ {error_msg}")
                return error_msg
            
            if verbose:
                print(f"\n🔧 Usando herramienta: {action.tool}")
                print(f"📥 Input: {action.tool_input}")
            
            observation = self.tools[action.tool].run(action.tool_input)
            
            if verbose:
                print(f"📤 Output: {observation}")
            
            # 5. Actualizar query con observación
            query = f"""Pregunta original: {query}
Razonamiento: {action.reasoning}
Herramienta usada: {action.tool}
Resultado: {observation}

Por favor, proporciona la respuesta final al usuario basándote en esta información."""
        
        # Si llegamos al máximo de iteraciones
        final_prompt = self._build_prompt(query)
        final_response = self._call_llm(final_prompt)
        
        if verbose:
            print(f"\n✅ Respuesta final:\n{final_response}")
        
        return final_response

print("✅ Clase SimpleAgent implementada")


### Probemos nuestro agente


In [ ]:
# Crear agente con herramientas
agent = SimpleAgent(
    tools=[calculator_tool, time_tool],
    llm_backend="simulated"  # Cambiar a "openai" o "anthropic" si tienes API keys
)

# Probar con una pregunta matemática
result1 = agent.run("¿Cuánto es el 15% de 340?")
print(f"\n{'='*60}")
print(f"Resultado Final: {result1}")


In [ ]:
# Probar con pregunta de tiempo
result2 = agent.run("¿Qué hora es?")
print(f"\n{'='*60}")
print(f"Resultado Final: {result2}")


<a id="5-ejercicios-graded"></a>
## 5. 🎓 Ejercicios Prácticos Guiados (100 pts)

En esta sección implementarás las funciones core de un sistema agéntico desde cero. Cada ejercicio construye sobre el anterior, llevándote desde parsear respuestas básicas hasta implementar un agente completo con memoria.

### 📊 Sistema de Calificación

- **Total de puntos**: 100
- **Mínimo para aprobar**: 70
- **Ejercicios**:
  1. `parse_tool_response` (15 pts) - Parsear respuestas del LLM
  2. `execute_tool` (15 pts) - Ejecutar herramientas de forma segura
  3. `build_agent_prompt` (20 pts) - Construir prompts efectivos
  4. `simple_agent_step` (25 pts) - Implementar un paso del loop agéntico
  5. `agent_with_memory` (25 pts) - Agente con memoria conversacional

### 🧪 Cómo Usar el Autograder

```python
# 1. Implementa tu función donde dice GRADED FUNCTION
# 2. Corre la celda de código
# 3. Ejecuta el grader:
from tests.test_01_llm_agents import LLMAgentsGrader
grader = LLMAgentsGrader()
grader.test_parse_tool_response(parse_tool_response)
```

### 💡 Tips

- Lee cuidadosamente los docstrings y ejemplos
- Los tests verifican casos edge (inputs vacíos, formatos incorrectos, etc.)
- Puedes correr tests individuales o todos a la vez
- Si fallas un test, el mensaje de error te dará pistas

---


### Ejercicio 1: Crear agente simple (20 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE HERE` y `# END CODE HERE`
2. Ejecuta el test para verificar tu implementación


In [ ]:
def create_simple_agent(llm_backend, tools):
    """
    Crea una instancia de SimpleAgent configurada con herramientas.
    
    Args:
        llm_backend: String indicando backend ("openai", "anthropic", "simulated")
        tools: Lista de objetos SimpleTool
    
    Returns:
        Instancia de SimpleAgent configurada
    
    Ejemplo:
        >>> agent = create_simple_agent("simulated", [calculator_tool, time_tool])
        >>> type(agent).__name__
        'SimpleAgent'
    """
    # START CODE HERE (≈ 2-4 líneas)
    # Pista: Ya existe la clase SimpleAgent definida arriba
    # Solo necesitas crear una instancia con los parámetros dados
    agent = SimpleAgent(tools=tools, llm_backend=llm_backend)
    # END CODE HERE
    return agent

<details><summary>💡 Hint 1: Estructura general</summary>

Para crear el agente necesitas:
1. Usar la clase `SimpleAgent` que ya está definida en el notebook (celda 8)
2. Pasar los parámetros `tools` y `llm_backend` al constructor
3. Retornar la instancia creada

La clase SimpleAgent ya está implementada, solo necesitas instanciarla correctamente.

</details>

<details><summary>💡 Hint 2: Parámetros del constructor</summary>

La firma del constructor de SimpleAgent es:
```python
def __init__(self, tools: List[SimpleTool], llm_backend: str = "simulated"):
```

Por lo tanto, debes pasar:
- `tools`: como primer argumento o argumento nombrado `tools=`
- `llm_backend`: como segundo argumento o argumento nombrado `llm_backend=`

</details>

<details><summary>💡 Hint 3: Ejemplo de uso</summary>

```python
# Ejemplo de cómo crear una instancia
agent = SimpleAgent(tools=mi_lista_de_tools, llm_backend="simulated")
```

Aplica esto a la función `create_simple_agent` usando los parámetros que recibe.

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def create_simple_agent(llm_backend, tools):
    """
    Crea una instancia de SimpleAgent configurada con herramientas.
    
    Args:
        llm_backend: String indicando backend ("openai", "anthropic", "simulated")
        tools: Lista de objetos SimpleTool
    
    Returns:
        Instancia de SimpleAgent configurada
    """
    # Paso 1: Crear instancia de SimpleAgent con los parámetros dados
    agent = SimpleAgent(tools=tools, llm_backend=llm_backend)
    
    # Paso 2: Retornar la instancia
    return agent
```

**Explicación:**
- **Línea 1**: Creamos una instancia de `SimpleAgent` pasando la lista de herramientas y el backend del LLM
- **Línea 2**: Retornamos el agente configurado

La clase `SimpleAgent` (definida arriba en el notebook) maneja internamente:
- La inicialización del diccionario de herramientas
- La configuración del cliente LLM según el backend
- El historial de interacciones

**Por qué funciona:**
- `SimpleAgent.__init__()` acepta exactamente estos parámetros
- El agente queda listo para ejecutar queries con `.run()`

</details>

In [ ]:
# 🧪 Test de create_simple_agent
print('Testing create_simple_agent...')
print('-' * 50)

# Test case 1: Crear agente con herramientas simples
try:
    test_tools = [calculator_tool, time_tool]
    agent = create_simple_agent("simulated", test_tools)
    
    # Verificar que es una instancia de SimpleAgent
    assert isinstance(agent, SimpleAgent), f"Expected SimpleAgent, got {type(agent)}"
    
    # Verificar que tiene las herramientas correctas
    assert len(agent.tools) == 2, f"Expected 2 tools, got {len(agent.tools)}"
    assert "calculator" in agent.tools, "calculator tool not found"
    assert "get_time" in agent.tools, "get_time tool not found"
    
    # Verificar backend
    assert agent.llm_backend == "simulated", f"Expected 'simulated' backend, got {agent.llm_backend}"
    
    print('✅ Test 1 passed: Agent created with 2 tools')
    print(f'   Tools: {list(agent.tools.keys())}')
    
except AssertionError as e:
    print(f'❌ Test 1 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 1: {e}')

# Test case 2: Agente con solo una herramienta
try:
    single_tool = [calculator_tool]
    agent = create_simple_agent("simulated", single_tool)
    
    assert len(agent.tools) == 1, f"Expected 1 tool, got {len(agent.tools)}"
    print('✅ Test 2 passed: Agent created with 1 tool')
    
except AssertionError as e:
    print(f'❌ Test 2 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 2: {e}')

# Test case 3: Verificar que el agente puede ejecutar
try:
    agent = create_simple_agent("simulated", [calculator_tool, time_tool])
    result = agent.run("¿Cuánto es 2 + 2?", verbose=False)
    
    assert result is not None, "Agent should return a result"
    assert isinstance(result, str), f"Expected string result, got {type(result)}"
    print('✅ Test 3 passed: Agent can execute queries')
    
except Exception as e:
    print(f'❌ Error en test 3: {e}')

print('\n✅ Todos los tests completados! +20 pts si todos pasaron')

### Ejercicio 2: Parsear output del LLM (20 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE HERE` y `# END CODE HERE`
2. Ejecuta el test para verificar tu implementación


In [ ]:
def parse_llm_output(output):
    """
    Parsea la respuesta del LLM para extraer acción y entrada.
    
    Busca patrones como:
    - "Action: nombre_accion"
    - "Action Input: input_para_accion"
    
    Args:
        output: String con la respuesta del LLM
    
    Returns:
        Dict con keys 'action' y 'action_input', o None si no se encuentra
    
    Ejemplo:
        >>> text = "Action: calculator\nAction Input: 2 + 2"
        >>> parse_llm_output(text)
        {'action': 'calculator', 'action_input': '2 + 2'}
    """
    # START CODE HERE (≈ 5-10 líneas)
    import re
    
    # Buscar patrón "Action: ..."
    action_match = re.search(r'Action:\s*(.+)', output, re.IGNORECASE)
    # Buscar patrón "Action Input: ..."
    input_match = re.search(r'Action Input:\s*(.+)', output, re.IGNORECASE)
    
    # Si encontramos ambos, retornar dict
    if action_match and input_match:
        return {
            'action': action_match.group(1).strip(),
            'action_input': input_match.group(1).strip()
        }
    
    # Si no encontramos el patrón completo, retornar None
    return None
    # END CODE HERE

<details><summary>💡 Hint 1: Uso de expresiones regulares</summary>

Necesitas buscar dos patrones en el texto:
1. `Action: <nombre_de_accion>` 
2. `Action Input: <entrada_para_accion>`

Usa el módulo `re` de Python para buscar estos patrones:
```python
import re
match = re.search(r'Action:\s*(.+)', text, re.IGNORECASE)
```

El `\s*` captura espacios opcionales, y `(.+)` captura el contenido.

</details>

<details><summary>💡 Hint 2: Extraer grupos de captura</summary>

Cuando encuentres un match con regex:
```python
if match:
    contenido = match.group(1).strip()  # group(1) es el primer ()
```

Necesitas hacer esto para AMBOS patrones:
- `Action:` → guarda en `action`
- `Action Input:` → guarda en `action_input`

</details>

<details><summary>💡 Hint 3: Manejo de casos sin match</summary>

Casos a considerar:
- ¿Qué pasa si solo encuentras "Action" pero no "Action Input"?
- ¿Qué pasa si el texto no tiene ninguno de los patrones?

Retorna `None` si no encuentras AMBOS patrones completos.

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def parse_llm_output(output):
    """
    Parsea la respuesta del LLM para extraer acción y entrada.
    """
    import re
    
    # Paso 1: Buscar patrón "Action: ..."
    action_match = re.search(r'Action:\s*(.+)', output, re.IGNORECASE)
    
    # Paso 2: Buscar patrón "Action Input: ..."
    input_match = re.search(r'Action Input:\s*(.+)', output, re.IGNORECASE)
    
    # Paso 3: Verificar que ambos existen
    if action_match and input_match:
        # Paso 4: Extraer y limpiar los valores
        return {
            'action': action_match.group(1).strip(),
            'action_input': input_match.group(1).strip()
        }
    
    # Paso 5: Si falta alguno, retornar None
    return None
```

**Explicación:**
- **Línea 1-2**: Usamos `re.search()` para buscar el patrón "Action:" seguido de cualquier texto
- **Línea 3-4**: Similar para "Action Input:"
- **Línea 5-6**: Solo retornamos dict si encontramos AMBOS patrones
- **Línea 7-10**: `.group(1)` extrae el contenido dentro de `()`, `.strip()` elimina espacios
- **Línea 11**: Si falta algún patrón, retornamos `None`

**Patrones Regex:**
- `r'Action:\s*(.+)'`: 
  - `Action:` → literal
  - `\s*` → cero o más espacios
  - `(.+)` → captura uno o más caracteres (el nombre de la acción)
- `re.IGNORECASE` → case insensitive (acepta "action" o "ACTION")

</details>

In [ ]:
# 🧪 Test de parse_llm_output
print('Testing parse_llm_output...')
print('-' * 50)

# Test case 1: Formato estándar
try:
    output1 = "Action: calculator\nAction Input: 15 * 340 / 100"
    result = parse_llm_output(output1)
    
    expected = {'action': 'calculator', 'action_input': '15 * 340 / 100'}
    assert result == expected, f"Expected {expected}, got {result}"
    print(f'✅ Test 1 passed: {result}')
    
except AssertionError as e:
    print(f'❌ Test 1 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 1: {e}')

# Test case 2: Con espacios extras y case insensitive
try:
    output2 = "ACTION:   get_time  \nACTION INPUT:   UTC   "
    result = parse_llm_output(output2)
    
    assert result is not None, "Should parse case-insensitive"
    assert result['action'] == 'get_time', f"Expected 'get_time', got {result['action']}"
    assert result['action_input'] == 'UTC', f"Expected 'UTC', got {result['action_input']}"
    print('✅ Test 2 passed: Handles case-insensitive and extra spaces')
    
except AssertionError as e:
    print(f'❌ Test 2 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 2: {e}')

# Test case 3: Input sin formato válido (falta Action Input)
try:
    output3 = "Action: search\nThis is just text without Action Input"
    result = parse_llm_output(output3)
    
    assert result is None, f"Should return None when Action Input is missing, got {result}"
    print('✅ Test 3 passed: Returns None for incomplete format')
    
except AssertionError as e:
    print(f'❌ Test 3 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 3: {e}')

# Test case 4: Texto sin ningún patrón
try:
    output4 = "This is just a plain response without any action"
    result = parse_llm_output(output4)
    
    assert result is None, f"Should return None for plain text, got {result}"
    print('✅ Test 4 passed: Returns None for plain text')
    
except AssertionError as e:
    print(f'❌ Test 4 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 4: {e}')

# Test case 5: Con texto adicional antes y después
try:
    output5 = """I need to calculate this.
Action: calculator
Action Input: 2 + 2
This should work."""
    result = parse_llm_output(output5)
    
    assert result is not None, "Should find pattern in multiline text"
    assert result['action'] == 'calculator'
    assert result['action_input'] == '2 + 2'
    print('✅ Test 5 passed: Finds pattern in multiline text')
    
except AssertionError as e:
    print(f'❌ Test 5 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 5: {e}')

print('\n✅ Todos los tests completados! +20 pts si todos pasaron')

### Ejercicio 3: Ejecutar tool (25 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE HERE` y `# END CODE HERE`
2. Ejecuta el test para verificar tu implementación


In [ ]:
def execute_tool(tool_name, tool_input, available_tools):
    """
    Ejecuta una herramienta por nombre con el input dado.
    
    Args:
        tool_name: Nombre de la herramienta a ejecutar
        tool_input: Input para la herramienta (string)
        available_tools: Dict de herramientas {nombre: SimpleTool}
    
    Returns:
        Resultado de la ejecución o mensaje de error
    
    Ejemplo:
        >>> tools = {"calculator": calculator_tool}
        >>> execute_tool("calculator", "2 + 2", tools)
        '4'
    """
    # START CODE HERE (≈ 4-8 líneas)
    # Verificar que la herramienta existe
    if tool_name not in available_tools:
        return f"Error: Herramienta '{tool_name}' no encontrada"
    
    # Obtener la herramienta
    tool = available_tools[tool_name]
    
    # Ejecutar la herramienta con el input
    result = tool.run(tool_input)
    
    return result
    # END CODE HERE

<details><summary>💡 Hint 1: Verificar existencia de la herramienta</summary>

Antes de ejecutar, verifica que `tool_name` existe en `available_tools`:
```python
if tool_name not in available_tools:
    return "Error: Herramienta no encontrada"
```

Los diccionarios de Python permiten verificar llaves con el operador `in`.

</details>

<details><summary>💡 Hint 2: Acceder y ejecutar la herramienta</summary>

Una vez verificado que existe:
```python
tool = available_tools[tool_name]  # Obtener el objeto SimpleTool
result = tool.run(tool_input)       # Ejecutar con el input
```

Recuerda que `SimpleTool` tiene un método `.run()` que acepta un string.

</details>

<details><summary>💡 Hint 3: Manejo de errores</summary>

La clase `SimpleTool.run()` ya maneja excepciones internamente y retorna strings de error.

Tu función solo necesita:
1. Verificar que la tool existe
2. Obtenerla del diccionario
3. Ejecutarla y retornar el resultado

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def execute_tool(tool_name, tool_input, available_tools):
    """
    Ejecuta una herramienta por nombre con el input dado.
    """
    # Paso 1: Verificar que la herramienta existe
    if tool_name not in available_tools:
        return f"Error: Herramienta '{tool_name}' no encontrada"
    
    # Paso 2: Obtener la herramienta del diccionario
    tool = available_tools[tool_name]
    
    # Paso 3: Ejecutar la herramienta con el input
    result = tool.run(tool_input)
    
    # Paso 4: Retornar el resultado
    return result
```

**Explicación:**
- **Línea 1-2**: Verificamos si `tool_name` está en las llaves del dict `available_tools`
- **Línea 3**: Si no existe, retornamos mensaje de error descriptivo
- **Línea 4**: Accedemos al objeto `SimpleTool` usando la llave
- **Línea 5**: Llamamos al método `.run(tool_input)` que ejecuta la función interna
- **Línea 6**: Retornamos el resultado (puede ser un valor o un error si la ejecución falló)

**Flujo de ejecución:**
```
available_tools = {"calculator": calculator_tool, "get_time": time_tool}

execute_tool("calculator", "2+2", available_tools)
  → tool_name="calculator" está en available_tools ✓
  → tool = calculator_tool (objeto SimpleTool)
  → result = tool.run("2+2")
    → calculator("2+2") 
    → eval("2+2") 
    → 4
  → return "4"
```

**Casos de error manejados:**
- Herramienta no existe → retorna "Error: Herramienta 'X' no encontrada"
- Error en ejecución → `SimpleTool.run()` ya maneja con try-except

</details>

In [ ]:
# 🧪 Test de execute_tool
print('Testing execute_tool...')
print('-' * 50)

# Preparar herramientas de prueba
test_tools = {
    "calculator": calculator_tool,
    "get_time": time_tool
}

# Test case 1: Ejecutar calculadora
try:
    result = execute_tool("calculator", "10 + 5", test_tools)
    expected = "15"
    assert result == expected, f"Expected '{expected}', got '{result}'"
    print(f'✅ Test 1 passed: calculator(10 + 5) = {result}')
    
except AssertionError as e:
    print(f'❌ Test 1 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 1: {e}')

# Test case 2: Ejecutar get_time
try:
    result = execute_tool("get_time", "UTC", test_tools)
    assert result is not None, "get_time should return a result"
    assert isinstance(result, str), f"Expected string, got {type(result)}"
    print(f'✅ Test 2 passed: get_time returned {result[:20]}...')
    
except AssertionError as e:
    print(f'❌ Test 2 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 2: {e}')

# Test case 3: Herramienta no existe
try:
    result = execute_tool("nonexistent_tool", "input", test_tools)
    assert "Error" in result, f"Should return error message, got {result}"
    assert "no encontrada" in result.lower() or "not found" in result.lower(), \
        f"Error message should mention tool not found, got '{result}'"
    print(f'✅ Test 3 passed: Handles missing tool')
    
except AssertionError as e:
    print(f'❌ Test 3 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 3: {e}')

# Test case 4: Input que causa error en la herramienta
try:
    result = execute_tool("calculator", "invalid expression!", test_tools)
    # SimpleTool.run maneja errores internamente
    assert result is not None, "Should return something even on error"
    print(f'✅ Test 4 passed: Handles tool execution errors')
    
except Exception as e:
    print(f'❌ Error en test 4: {e}')

# Test case 5: Operación matemática compleja
try:
    result = execute_tool("calculator", "2 ** 8", test_tools)  # 2^8 = 256
    expected = "256"
    assert result == expected, f"Expected '{expected}', got '{result}'"
    print(f'✅ Test 5 passed: calculator(2 ** 8) = {result}')
    
except AssertionError as e:
    print(f'❌ Test 5 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 5: {e}')

print('\n✅ Todos los tests completados! +25 pts si todos pasaron')

### Ejercicio 4: Loop del agente (20 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE HERE` y `# END CODE HERE`
2. Ejecuta el test para verificar tu implementación


In [ ]:
def agent_loop(query, tools, max_iterations=3):
    """
    Implementa el loop básico de un agente: Think → Act → Observe.
    
    Args:
        query: Pregunta del usuario
        tools: Dict de herramientas disponibles
        max_iterations: Máximo número de iteraciones
    
    Returns:
        String con la respuesta final
    
    Ejemplo:
        >>> tools = {"calculator": calculator_tool}
        >>> agent_loop("¿Cuánto es 5 + 3?", tools)
        'Respuesta: 8'
    """
    # START CODE HERE (≈ 10-15 líneas)
    history = []
    
    for i in range(max_iterations):
        # Think: Decidir qué hacer (simplificado para demo)
        if i == 0:
            thought = f"Necesito procesar: {query}"
            action = "calculator" if "cuánto" in query.lower() or "+" in query or "-" in query else None
        else:
            thought = "Ya tengo suficiente información"
            action = None
        
        history.append(f"Thought {i+1}: {thought}")
        
        # Act: Ejecutar acción si es necesaria
        if action and action in tools:
            # Extraer expresión numérica simple (demo)
            import re
            expr_match = re.search(r'[\d\s+\-*/()]+', query)
            expr = expr_match.group(0) if expr_match else "1+1"
            
            observation = execute_tool(action, expr, tools)
            history.append(f"Action {i+1}: {action}[{expr}]")
            history.append(f"Observation {i+1}: {observation}")
            
            # Si obtuvimos resultado, terminar
            return f"Respuesta: {observation}"
        else:
            # No hay más acciones, dar respuesta directa
            return f"Respuesta basada en razonamiento: {query}"
    
    return "No pude completar la tarea en el límite de iteraciones"
    # END CODE HERE

<details><summary>💡 Hint 1: Estructura del loop</summary>

El loop agéntico básico tiene 3 fases en cada iteración:
```python
for i in range(max_iterations):
    # 1. THINK: Razonar sobre qué hacer
    thought = decidir_siguiente_paso(query, history)
    
    # 2. ACT: Ejecutar acción si es necesaria
    if necesita_herramienta:
        observation = execute_tool(...)
    
    # 3. OBSERVE: Procesar resultado y decidir si continuar
    if tarea_completa:
        return respuesta_final
```

</details>

<details><summary>💡 Hint 2: Decisión de acciones</summary>

Para simplificar el ejercicio, puedes usar heurísticas básicas:
- Si la query contiene "cuánto", "+", "-", etc. → usar calculator
- Si la query contiene "hora", "fecha" → usar get_time  
- Si ya tienes resultado → terminar

En un agente real, esto lo decide el LLM.

</details>

<details><summary>💡 Hint 3: Criterio de parada</summary>

El loop debe terminar cuando:
1. Se obtiene una respuesta satisfactoria (después de ejecutar herramienta)
2. No se necesitan más herramientas
3. Se alcanza `max_iterations`

Usa `return` para salir del loop temprano.

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def agent_loop(query, tools, max_iterations=3):
    """
    Implementa el loop básico de un agente: Think → Act → Observe.
    """
    import re
    history = []
    
    for i in range(max_iterations):
        # FASE 1: THINK - Decidir qué hacer
        if i == 0:
            thought = f"Necesito procesar: {query}"
            # Heurística simple: detectar si necesita cálculo
            action = "calculator" if any(word in query.lower() for word in ["cuánto", "+", "-", "*", "/"]) else None
        else:
            thought = "Ya tengo suficiente información"
            action = None
        
        history.append(f"Thought {i+1}: {thought}")
        
        # FASE 2: ACT - Ejecutar acción si es necesaria
        if action and action in tools:
            # Extraer expresión numérica (simplificado)
            expr_match = re.search(r'[\d\s+\-*/()]+', query)
            expr = expr_match.group(0).strip() if expr_match else "1+1"
            
            # Ejecutar herramienta
            observation = execute_tool(action, expr, tools)
            history.append(f"Action {i+1}: {action}[{expr}]")
            history.append(f"Observation {i+1}: {observation}")
            
            # FASE 3: OBSERVE - Evaluar si terminamos
            return f"Respuesta: {observation}"
        else:
            # No hay más acciones necesarias
            return f"Respuesta basada en razonamiento: {query}"
    
    # Si llegamos aquí, se agotaron las iteraciones
    return "No pude completar la tarea en el límite de iteraciones"
```

**Explicación del flujo:**

1. **Inicialización**: Creamos `history` para trackear el proceso

2. **Loop principal** (`for i in range(max_iterations)`):
   - **Iteración 0**: Analiza la query y decide acción
   - **Iteración 1+**: Ya debería tener respuesta

3. **THINK fase**:
   - Analiza qué hacer basado en la query
   - Usa heurísticas simples (en producción, usar LLM)
   - Genera un `thought` explícito

4. **ACT fase**:
   - Si identificó acción necesaria, extrae parámetros
   - Ejecuta la herramienta con `execute_tool()`
   - Registra la acción en history

5. **OBSERVE fase**:
   - Evalúa el resultado (observation)
   - Si es satisfactorio, retorna respuesta final
   - Si no, continúa al siguiente loop

6. **Criterios de parada**:
   - ✅ Obtiene respuesta de herramienta → termina
   - ✅ No necesita herramientas → responde directamente
   - ⚠️  Max iterations → mensaje de error

**Ejemplo de ejecución**:
```
query = "¿Cuánto es 10 + 5?"

Iteración 0:
  Thought: "Necesito procesar: ¿Cuánto es 10 + 5?"
  Action: calculator[10 + 5]
  Observation: "15"
  → return "Respuesta: 15"
```

</details>

In [ ]:
# 🧪 Test de agent_loop
print('Testing agent_loop...')
print('-' * 50)

test_tools = {"calculator": calculator_tool, "get_time": time_tool}

# Test case 1: Query que requiere calculator
try:
    result = agent_loop("¿Cuánto es 25 + 17?", test_tools)
    
    assert "Respuesta" in result, f"Should contain 'Respuesta', got '{result}'"
    assert "42" in result, f"Expected '42' in result, got '{result}'"
    print(f'✅ Test 1 passed: {result}')
    
except AssertionError as e:
    print(f'❌ Test 1 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 1: {e}')

# Test case 2: Query que no requiere herramientas
try:
    result = agent_loop("Hola, ¿cómo estás?", test_tools)
    
    assert result is not None, "Should return something"
    assert isinstance(result, str), f"Expected string, got {type(result)}"
    print(f'✅ Test 2 passed: Handles non-tool query')
    
except Exception as e:
    print(f'❌ Error en test 2: {e}')

# Test case 3: Operación matemática compleja
try:
    result = agent_loop("Calcula 100 - 45", test_tools)
    
    assert "Respuesta" in result, "Should contain 'Respuesta'"
    assert "55" in result, f"Expected '55', got '{result}'"
    print(f'✅ Test 3 passed: Complex calculation')
    
except AssertionError as e:
    print(f'❌ Test 3 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 3: {e}')

print('\n✅ Todos los tests completados! +20 pts si todos pasaron')

### Ejercicio 5: Formatear prompt (15 pts)

**Instrucciones:**
1. Completa el código entre `# START CODE HERE` y `# END CODE HERE`
2. Ejecuta el test para verificar tu implementación


In [ ]:
def format_prompt(query, history, tools):
    """
    Construye un prompt para el LLM con template, historial y herramientas.
    
    Args:
        query: Pregunta del usuario (string)
        history: Lista de interacciones pasadas
        tools: Dict de herramientas {nombre: SimpleTool}
    
    Returns:
        String con el prompt formateado
    
    Ejemplo:
        >>> tools = {"calc": calculator_tool}
        >>> prompt = format_prompt("¿Cuánto es 2+2?", [], tools)
        >>> "Herramientas disponibles" in prompt
        True
    """
    # START CODE HERE (≈ 8-12 líneas)
    # Construir descripción de herramientas
    tools_desc = []
    for name, tool in tools.items():
        tools_desc.append(f"- {name}: {tool.description}")
    tools_text = "\n".join(tools_desc)
    
    # Construir historial
    history_text = "\n".join(history) if history else "Sin historial previo"
    
    # Template del prompt
    prompt = f"""Eres un asistente útil que puede usar herramientas para responder preguntas.

Herramientas disponibles:
{tools_text}

Para usar una herramienta, responde en formato:
Action: nombre_herramienta
Action Input: entrada_para_herramienta

Historial:
{history_text}

Pregunta del usuario: {query}

Tu respuesta:"""
    
    return prompt
    # END CODE HERE

<details><summary>💡 Hint 1: Estructura del prompt</summary>

Un prompt para agente típicamente incluye:
```python
prompt = f"""Sistema: Eres un asistente...

Herramientas: {lista_de_tools}

Historial: {conversacion_previa}

Query: {pregunta_actual}

Responde:"""
```

</details>

<details><summary>💡 Hint 2: Formatear herramientas</summary>

Itera sobre el diccionario `tools` y construye una descripción:
```python
tools_desc = []
for name, tool in tools.items():
    tools_desc.append(f"- {name}: {tool.description}")
tools_text = "\n".join(tools_desc)
```

</details>

<details><summary>💡 Hint 3: Incluir instrucciones</summary>

El prompt debe indicar al LLM cómo usar las herramientas:
```
Para usar una herramienta, responde:
Action: nombre_tool
Action Input: parametro
```

Esto le enseña el formato que debe seguir.

</details>

<details><summary>🔑 Solución Completa</summary>

```python
def format_prompt(query, history, tools):
    """
    Construye un prompt para el LLM con template, historial y herramientas.
    """
    # Paso 1: Construir descripción de herramientas
    tools_desc = []
    for name, tool in tools.items():
        tools_desc.append(f"- {name}: {tool.description}")
    tools_text = "\n".join(tools_desc)
    
    # Paso 2: Formatear historial
    history_text = "\n".join(history) if history else "Sin historial previo"
    
    # Paso 3: Construir prompt completo con template
    prompt = f"""Eres un asistente útil que puede usar herramientas para responder preguntas.

Herramientas disponibles:
{tools_text}

Para usar una herramienta, responde en formato:
Action: nombre_herramienta
Action Input: entrada_para_herramienta

Historial:
{history_text}

Pregunta del usuario: {query}

Tu respuesta:"""
    
    return prompt
```

**Explicación:**

1. **Descripción de herramientas** (Paso 1):
   - Iteramos sobre `tools.items()` para obtener (nombre, objeto)
   - Para cada tool, creamos línea: `"- nombre: descripción"`
   - Unimos todo con saltos de línea

2. **Formatear historial** (Paso 2):
   - Si hay historial, lo unimos con `\n`
   - Si no hay, mensaje por defecto
   - Esto permite al LLM ver contexto previo

3. **Template completo** (Paso 3):
   - **System message**: Define el rol del asistente
   - **Herramientas**: Lista lo que puede usar
   - **Instrucciones**: Formato de respuesta esperado
   - **Historial**: Contexto de conversación
   - **Query actual**: La pregunta a responder
   - **Prompt de cierre**: "Tu respuesta:" invita a completar

**Por qué este diseño:**
- **Claridad**: LLM sabe exactamente qué puede hacer
- **Consistencia**: Formato estructurado facilita parsing
- **Contexto**: Historial previene repetición
- **Instrucciones explícitas**: Mejora adherencia al formato

**Ejemplo de output**:
```
Eres un asistente útil que puede usar herramientas para responder preguntas.

Herramientas disponibles:
- calculator: Calcula expresiones matemáticas
- get_time: Retorna fecha y hora actual

Para usar una herramienta, responde en formato:
Action: nombre_herramienta
Action Input: entrada_para_herramienta

Historial:
Sin historial previo

Pregunta del usuario: ¿Cuánto es 15 + 27?

Tu respuesta:
```

</details>

In [ ]:
# 🧪 Test de format_prompt
print('Testing format_prompt...')
print('-' * 50)

test_tools = {"calculator": calculator_tool, "get_time": time_tool}

# Test case 1: Prompt sin historial
try:
    prompt = format_prompt("¿Cuánto es 5 + 3?", [], test_tools)
    
    assert isinstance(prompt, str), f"Expected string, got {type(prompt)}"
    assert "Herramientas disponibles" in prompt, "Should mention available tools"
    assert "calculator" in prompt, "Should list calculator tool"
    assert "get_time" in prompt, "Should list get_time tool"
    assert "¿Cuánto es 5 + 3?" in prompt, "Should include user query"
    print('✅ Test 1 passed: Basic prompt structure')
    
except AssertionError as e:
    print(f'❌ Test 1 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 1: {e}')

# Test case 2: Prompt con historial
try:
    history = ["Previous: Calculé 2+2=4", "Previous: La hora era 10:30"]
    prompt = format_prompt("Nueva pregunta", history, test_tools)
    
    assert "Previous: Calculé 2+2=4" in prompt, "Should include history"
    assert "Previous: La hora era 10:30" in prompt, "Should include full history"
    print('✅ Test 2 passed: Includes history')
    
except AssertionError as e:
    print(f'❌ Test 2 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 2: {e}')

# Test case 3: Verificar instrucciones de formato
try:
    prompt = format_prompt("Test query", [], test_tools)
    
    assert "Action:" in prompt, "Should include Action format instruction"
    assert "Action Input:" in prompt, "Should include Action Input format instruction"
    print('✅ Test 3 passed: Includes format instructions')
    
except AssertionError as e:
    print(f'❌ Test 3 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 3: {e}')

# Test case 4: Verificar descripciones de herramientas
try:
    prompt = format_prompt("Query", [], test_tools)
    
    # Verificar que incluye las descripciones de las tools
    assert "expresiones matemáticas" in prompt.lower() or "cálculos" in prompt.lower(), \
        "Should include calculator description"
    assert "hora" in prompt.lower() or "fecha" in prompt.lower(), \
        "Should include time tool description"
    print('✅ Test 4 passed: Includes tool descriptions')
    
except AssertionError as e:
    print(f'❌ Test 4 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 4: {e}')

# Test case 5: Prompt con una sola herramienta
try:
    single_tool = {"calculator": calculator_tool}
    prompt = format_prompt("Calculate this", [], single_tool)
    
    assert "calculator" in prompt, "Should include the single tool"
    assert "get_time" not in prompt, "Should not include missing tools"
    print('✅ Test 5 passed: Handles single tool')
    
except AssertionError as e:
    print(f'❌ Test 5 failed: {e}')
except Exception as e:
    print(f'❌ Error en test 5: {e}')

print('\n✅ Todos los tests completados! +15 pts si todos pasaron')
print('\n🎉 NOTEBOOK 01 COMPLETADO - 100/100 pts posibles')